### Few shot prompt engernering

In [ ]:
!pip install -q -U trl==0.12 transformers accelerate
!pip install -q -U datasets bitsandbytes

^C


In [7]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

model_name = 'cjvt/GaMS-9B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # loading in 4 bit
    bnb_4bit_quant_type="nf4", # quantization type
    bnb_4bit_use_double_quant=True, # nested quantization
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/4 [00:11<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.71 GiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 26.31 GiB is allocated by PyTorch, and 10.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Few Shot prompt engeneering

In [13]:
generator = pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    temperature=0.001,
    max_new_tokens=500,
    repetition_penalty=1.1
)

Device set to use cuda:0


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.10 GiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 26.31 GiB is allocated by PyTorch, and 10.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [10]:
import pandas as pd

REPORTS = pd.read_csv("traffic_reports_linked.csv", encoding='utf-8')
#REPORTS.head()

In [11]:
RTFS = pd.read_csv("rtfs_reduced.csv", encoding='utf-8')
#RTFS.head()

In [14]:
from sklearn.model_selection import train_test_split

RTFS_train, temp = train_test_split(RTFS, test_size=0.3, random_state=42)
RTFS_valid, RTFS_test = train_test_split(temp, test_size=0.5, random_state=42)
print(f"RTFS size: {len(RTFS)}")
print(f"RTFS_train size: {len(RTFS_train)}")
print(f"RTFS_valid size: {len(RTFS_valid)}")
print(f"RTFS_test size: {len(RTFS_test)}")

RTFS size: 28037
RTFS_train size: 19625
RTFS_valid size: 4206
RTFS_test size: 4206


In [48]:
def prepare_input(RTF_file_name):

    # Traffic data for the given RTF file name
    reports = REPORTS[REPORTS['RTF_file_name'] == RTF_file_name]

    # Input data used for generating desired LLM output
    input = {col: set(reports[col].dropna()) for col in [
        'Datum',
        'A1', 
        'B1', 
        'ContentPomembnoSLO', 
        'ContentNesreceSLO', 
        'ContentZastojiSLO', 
        'ContentVremeSLO', 
        'ContentOvireSLO', 
        'ContentDeloNaCestiSLO', 
        'ContentOpozorilaSLO',
        'ContentMednarodneInformacijeSLO', 
        'ContentSplosnoSLO']}
    
    lines = ["### Vhodni podatki:"]
    
    if input['ContentPomembnoSLO']:
        lines.append(f"- Zelo pomembne informacije o prometu: {', '.join(map(str, input['ContentPomembnoSLO']))}")
    if input['A1']:
        lines.append(f"- Pomembne informacije o prometu: {', '.join(map(str, input['A1']))}")
    if input['B1']:
        lines.append(f"- Manj pomembne informacije o prometu: {', '.join(map(str, input['B1']))}")
    if input['ContentNesreceSLO']:
        lines.append(f"- Informacije o nesrečah: {', '.join(map(str, input['ContentNesreceSLO']))}")
    if input['ContentZastojiSLO']:
        lines.append(f"- Informacije o zastojih: {', '.join(map(str, input['ContentZastojiSLO']))}")
    if input['ContentVremeSLO']:
        lines.append(f"- Informacije o vremenu: {', '.join(map(str, input['ContentVremeSLO']))}")
    if input['ContentOvireSLO']:
        lines.append(f"- Informacije o ovirah: {', '.join(map(str, input['ContentOvireSLO']))}")
    if input['ContentDeloNaCestiSLO']:
        lines.append(f"- Informacije o delu na cesti: {', '.join(map(str, input['ContentDeloNaCestiSLO']))}")
    if input['ContentOpozorilaSLO']:
        lines.append(f"- Informacije o opozorilih: {', '.join(map(str, input['ContentOpozorilaSLO']))}")
    if input['ContentMednarodneInformacijeSLO']:
        lines.append(f"- Informacije o mednarodnih informacijah: {', '.join(map(str, input['ContentMednarodneInformacijeSLO']))}")
    if input['ContentSplosnoSLO']:
        lines.append(f"- Splošne informacije: {', '.join(map(str, input['ContentSplosnoSLO']))}")
    
    if lines == ["### Vhodni podatki:"]:
        return None

    return lines

def generate_shot(RTFS_, RTF_file_name):
    
    # Preparing example shot text
    shot = prepare_input(RTF_file_name)

    if shot == None:
        return None

    # Desired LLM output
    output = RTFS_[RTFS_['file_name'] == RTF_file_name]["content"].values[0]

    shot.append("")
    shot.append(f"### Poročilo: {output}\n")

    return '\n'.join(shot)

# Example usage
print(generate_shot(RTFS_train, 2))


### Vhodni podatki:
- Manj pomembne informacije o prometu: Na primorski avtocesti med Vrhniko in Logatcem proti Kopru zaprt počasni pas zaradi okvare vozila.Zastoj proti Avstriji je na gorenjski avtocesti pred predorom Karavanke, 3 km. Proti Kranjski Gori priporočamo že izvoz Jesenice vzhod.Čakalna doba je na Obrežju in Gruškovju.Predvidene popolne zapore ta konec tedna:- bo do ponedeljka do 5. ure zaprt zavijalni pas na priključku Ljubljana sever iz smeri Bežigrada proti Medvodam in zavijalni pas iz smeri Medvod proti Kosezam.- , v Purgi do 17.30.- , v Domžalah na Kamniški cesti bo zaprt podvoz do 19. ure.- , v Družinski vasi, do 19. ure.- , pri Orehku, do nedelje do 19. ure.Na cesti Kamnik - Stahovica - Gornji Grad bo do 26. aprila zaprt odsek Črna pri Kamniku - Potok v Črni.Popolni zapori:- Jurovski dol - Lenart, v nedeljo od 8.30 do 14. ure.
- Informacije o zastojih: Zastoj proti Avstriji je na gorenjski avtocesti pred predorom Karavanke, 3 km. Proti Kranjski Gori priporočamo že iz

In [49]:
# Randomly select a test RTF file name and prepare the input data
random_test_RTF = RTFS_test.sample(1).iloc[0]['file_name']

# Ipnut data lines for the test sample
test_data = '\n'.join(prepare_input(random_test_RTF))

# Generate the few-shot prompt with some training examples
few_shot_prompt = f"""Generiraj prometno poročilo na podlagi spodnjih vhodnih podatkov:
{test_data}

Spodaj je podanih par primerov vhodnih poročil in željenih izhodnih poročil iz teh podatkov:

{generate_shot(RTFS_train, 1)}
{generate_shot(RTFS_train, 2)}
{generate_shot(RTFS_train, 3)}
"""
# Example of few-shot prompt
#print(few_shot_prompt)
with open("few_shot_prompt.txt", "w", encoding="utf-8") as f:
    f.write(few_shot_prompt)

In [18]:
res = generator(few_shot_prompt)
print(res[0]["generated_text"])

NameError: name 'generator' is not defined

Generate dataset for training, testing and validation (prepare prompts).

In [50]:
from datasets import Dataset

RTFS_train["prompt"] = None
RTFS_test["prompt"] = None
RTFS_valid["prompt"] = None
for index, row in RTFS_train.iterrows():
    shot = generate_shot(RTFS_train, row['file_name'])
    if shot:
        RTFS_train.at[index, 'prompt'] =  shot

for index, row in RTFS_test.iterrows():
    shot = generate_shot(RTFS_test, row['file_name'])
    if shot:
        RTFS_test.at[index, 'prompt'] =  shot

for index, row in RTFS_valid.iterrows():
    shot = generate_shot(RTFS_valid, row['file_name'])
    if shot:
        RTFS_valid.at[index, 'prompt'] =  shot

train_dataset = RTFS_train[RTFS_train["prompt"].notnull()]["prompt"]
test_dataset = RTFS_test[RTFS_test["prompt"].notnull()]["prompt"]
valid_dataset = RTFS_valid[RTFS_valid["prompt"].notnull()]["prompt"]

train_dataset = Dataset.from_pandas(train_dataset)
test_dataset = Dataset.from_pandas(test_dataset)
valid_dataset = Dataset.from_pandas(valid_dataset)



AttributeError: 'Series' object has no attribute 'columns'

# Low_Rank adaptation (LoRA)

In [19]:
from peft import LoraConfig, get_peft_model

lora_alpha = 32
lora_dropout = 0.1
lora_r = 16

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM"
)

In [20]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [21]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [22]:
lora_model = get_peft_model(model, peft_config)

In [23]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_

In [24]:
print_trainable_parameters(lora_model)

trainable params: 3194880 || all params: 3207360768 || trainable%: 0.0996108710898842


## Loading the trainer

In [ ]:
from transformers import TrainingArguments
output_dir = "./results"
per_device_train_batch_size = 1
gradient_accumulation_steps = 1
optim = "paged_adamw_32bit" #specialization of the AdamW optimizer that enables efficient learning in LoRA setting.
save_steps = 100
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 500
warmup_ratio = 0.03
lr_scheduler_type = "constant"

In [26]:
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    report_to="none"
)

In [ ]:
from trl import SFTTrainer

max_seq_length = 512

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
)

NameError: name 'dataset' is not defined

In [28]:
model_to_save = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model  # Take care of distributed/parallel training
model_to_save.save_pretrained("outputs")

trainer.train()

NameError: name 'trainer' is not defined

The `SFTTrainer` also takes care of properly saving only the adapters during training instead of saving the entire model.

In [ ]:
#model_to_save = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model  # Take care of distributed/parallel training
#model_to_save.save_pretrained("outputs")

# Loading the Adapater (the residual)

In [ ]:
lora_config = LoraConfig.from_pretrained('outputs')
model = get_peft_model(model, lora_config)